In [1]:
import pandas as pd
# https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_OHXDEN.XPT
df = pd.read_sas("../data/raw/P_OHXDEN.XPT")
ctc_cols = [c for c in df.columns if c.endswith('CTC')]
has_untreated_coronal = (
    df[ctc_cols]
    .apply(lambda row: (row == b'U').any(), axis=1)
)
has_root_caries = df["OHXRCAR"] == 1
df["has_caries"] = (has_root_caries | has_untreated_coronal).astype(int)
tc_cols = [c for c in df.columns if c.endswith("TC") and not c.endswith("CTC")]
df["n_missing_teeth"] = (df[tc_cols] == 4).sum(axis=1)
df["n_filled_teeth"] = (df[ctc_cols] == b'F').sum(axis=1)
df["n_untreated_teeth"] = (df[ctc_cols] == b'U').sum(axis=1)
df["has_root_caries"] = (df["OHXRCAR"] == 1).astype(int)
features = [
    "SEQN",
    "has_caries",
    "n_missing_teeth",
    "n_filled_teeth",
    "n_untreated_teeth",
    "has_root_caries",
]

df = df[features]

# https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_DEMO.XPT
demo_df = pd.read_sas("../data/raw/P_DEMO.XPT")

demo_df = demo_df[
    [
        "SEQN",
        "RIDAGEYR",
        "RIAGENDR",
        "INDFMPIR"   # ← SES feature
    ]
]

demo_df["is_female"] = (demo_df["RIAGENDR"] == 2).astype(int)
demo_df = demo_df.drop("RIAGENDR", axis=1)

df = df.merge(demo_df, on="SEQN", how="inner")
df["INDFMPIR"] = df["INDFMPIR"].fillna(df["INDFMPIR"].median())

# https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2017/DataFiles/P_SMQ.XPT
smoker_df = pd.read_sas("../data/raw/P_SMQ.XPT")
smoker_df = smoker_df[['SEQN', 'SMQ020']]
smoker_df['ever_smoked'] = (smoker_df['SMQ020']==1).astype(int)
df = df.merge(smoker_df, on='SEQN', how='inner')

features = [
        "RIDAGEYR",
        "is_female",
        "n_missing_teeth",
        "n_filled_teeth",
        "ever_smoked",
        "INDFMPIR",
        "has_root_caries"
]

df_model = df[features]

df_model.to_parquet("../data/processed/final_caries_features.parquet", index=False)